# Lab 1 — Supervised Fine-Tuning: contract review that cites its evidence

## Notebook 4 — Deploy to a SageMaker real-time endpoint

Two ways to serve a customized model, and they suit different demand shapes. This
notebook deploys the first:

| | **SageMaker real-time endpoint** | **Bedrock Custom Model Import** |
|---|---|---|
| infrastructure | dedicated GPU instance you choose | fully serverless |
| billing | per instance-hour, 24/7 while it exists | per Custom Model Unit in 5-minute active windows |
| idle cost | **you keep paying** | scales to zero |
| cold start | none once InService | tens of seconds after idle |
| best for | steady high throughput | spiky or low-volume workloads |

Read the right-hand column as the trade you are making by choosing an endpoint: you
pay for idle time in exchange for no cold start.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")

In [ ]:
import hashlib

from config import BASE_MODEL_ID

base_model_id = BASE_MODEL_ID

# SageMaker caps Model Package Group names at 63 characters. Hash-truncate when
# base_model_id + suffix exceeds it, so notebooks 2, 3 and 4 all derive the
# same name for any model id.
MAX_MPG_NAME_LENGTH = 63
suffix = "-contractnli-sft-mpg"

candidate = f"{base_model_id}{suffix}"
if len(candidate) > MAX_MPG_NAME_LENGTH:
    digest = hashlib.sha1(base_model_id.encode()).hexdigest()[:6]
    # reserve room for the suffix, a hyphen separator, and the 6-char hash
    keep = MAX_MPG_NAME_LENGTH - len(suffix) - len(digest) - 1
    truncated = base_model_id[:keep].rstrip("-")
    model_package_group_name = f"{truncated}-{digest}{suffix}"
else:
    model_package_group_name = candidate

print(f"Model Package Group: {model_package_group_name}")


In [ ]:
from sagemaker.core import s3
from sagemaker.core.resources import ModelPackage

resp = sm_client.list_model_packages(
    ModelPackageGroupName=model_package_group_name,
    SortBy="CreationTime", SortOrder="Descending", MaxResults=1)
assert resp["ModelPackageSummaryList"], "no model packages found - run notebook 2 first"

model_package = ModelPackage.get(resp["ModelPackageSummaryList"][0]["ModelPackageArn"])

# the *merged* checkpoint (base weights + LoRA applied) is what we deploy
merged_model_s3_uri = s3.s3_path_join(
    model_package.inference_specification.containers[0]
    .model_data_source.s3_data_source.s3_uri,
    "checkpoints", "hf_merged") + "/"

print(f"merged model: {merged_model_s3_uri}")

Create the custom nano_v3 reasoning parser plugin and upload it to S3

In [ ]:
reasoning_parser_file = "nano_v3_reasoning_parser.py"

nano_v3_reasoning_parser = '''from vllm.reasoning.abs_reasoning_parsers import ReasoningParserManager
from vllm.reasoning.deepseek_r1_reasoning_parser import DeepSeekR1ReasoningParser


@ReasoningParserManager.register_module("nano_v3")
class NanoV3ReasoningParser(DeepSeekR1ReasoningParser):
    def extract_reasoning(self, model_output, request):
        reasoning_content, final_content = super().extract_reasoning(
            model_output, request
        )
        if (
            hasattr(request, "chat_template_kwargs")
            and request.chat_template_kwargs
            and request.chat_template_kwargs.get("enable_thinking") is False
            and final_content is None
        ):
            reasoning_content, final_content = final_content, reasoning_content

        return reasoning_content, final_content
'''

with open(reasoning_parser_file, "w") as f:
    f.write(nano_v3_reasoning_parser)

Create vLLM configurations for tool parser

In [ ]:
vllm_config_file = "vllm_config.yaml"

vllm_config = """enable_auto_tool_choice: true
tool_call_parser: qwen3_coder
reasoning_parser: nano_v3
reasoning_parser_plugin: /opt/ml/model/nano_v3_reasoning_parser.py
"""

with open(vllm_config_file, "w") as f:
    f.write(vllm_config)

Upload files in the S3 model path

In [ ]:
import os
from urllib.parse import urlparse

parsed = urlparse(merged_model_s3_uri)
model_bucket = parsed.netloc
model_prefix = parsed.path.lstrip("/").rstrip("/")

s3_client.upload_file(
    reasoning_parser_file, model_bucket, f"{model_prefix}/{reasoning_parser_file}"
)
s3_client.upload_file(
    vllm_config_file, model_bucket, f"{model_prefix}/{vllm_config_file}"
)

os.remove(reasoning_parser_file)
os.remove(vllm_config_file)

print("Uploaded vLLM config + reasoning parser to:")
print(f"s3://{model_bucket}/{model_prefix}/{reasoning_parser_file}")
print(f"s3://{model_bucket}/{model_prefix}/{vllm_config_file}")

### Resource names

In [ ]:
import hashlib

MAX_NAME = 63


def rname(base, suffix):
    cand = f"{base}{suffix}"
    if len(cand) <= MAX_NAME:
        return cand
    digest = hashlib.sha1(base.encode()).hexdigest()[:6]
    keep = MAX_NAME - len(suffix) - len(digest) - 1
    return f"{base[:keep].rstrip('-')}-{digest}{suffix}"


stem = f"{base_model_id}-contractnli"
model_name = rname(stem, "-sft-m")
endpoint_config_name = rname(stem, "-sft-cfg")
endpoint_name = rname(stem, "-sft-ep")
ic_name = rname(stem, "-sft-ic")
print(model_name, endpoint_config_name, endpoint_name, ic_name, sep="\n")

### Create the model, endpoint config and endpoint

Specify Instance type, count, and number of GPUs per instance

In [ ]:
instance_count = 1
instance_type = "ml.g5.12xlarge"
number_of_gpu = 4  # ml.g5.12xlarge has 4x A10G GPUs -> vLLM tensor parallel size
health_check_timeout = 700

In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig
from sagemaker.core.shapes import ProductionVariant

try:
    EndpointConfig.get(endpoint_config_name)
    print(f"endpoint config exists: {endpoint_config_name}")
except Exception:
    EndpointConfig.create(
        endpoint_config_name=endpoint_config_name,
        execution_role_arn=role,
        production_variants=[ProductionVariant(
            variant_name="AllTraffic",
            instance_type=instance_type,
            initial_instance_count=1,
            model_data_download_timeout_in_seconds=health_check_timeout,
            routing_config={"routing_strategy": "LEAST_OUTSTANDING_REQUESTS"})])
    print(f"created endpoint config: {endpoint_config_name}")

### Create Endpoint

A SageMaker Endpoint is a fully managed, always-on HTTPS API that hosts your deployed model and serves real-time inference requests.

In [ ]:
try:
    endpoint = Endpoint.get(endpoint_name)
    print(f"endpoint exists: {endpoint_name}")
except Exception:
    endpoint = Endpoint.create(endpoint_name=endpoint_name,
                               endpoint_config_name=endpoint_config_name)
    print(f"creating endpoint: {endpoint_name}")

endpoint.wait_for_status("InService")
print("endpoint InService")

### Create the Model

Define vLLM configurations

In [ ]:
import json

env = {
    "SM_VLLM_MODEL": "/opt/ml/model",  # path where SageMaker mounts the model
    "SM_VLLM_CONFIG": "/opt/ml/model/vllm_config.yaml",
    "SM_VLLM_DTYPE": "bfloat16",
    "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.9",
    "SM_VLLM_MAX_MODEL_LEN": json.dumps(1024 * 32),
    "SM_VLLM_MAX_NUM_SEQS": "16",
    "SM_VLLM_ENABLE_CHUNKED_PREFILL": "true",
    "SM_VLLM_KV_CACHE_DTYPE": "auto",
    "SM_VLLM_TENSOR_PARALLEL_SIZE": str(number_of_gpu),
}

Specify the vLLM container image URI

In [ ]:
region = sess.boto_region_name
CONTAINER_VERSION = "vllm:0.22.0-gpu-py312-cu130-ubuntu22.04-sagemaker"

inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/{CONTAINER_VERSION}"

print(inference_image)

Create the resources

In [ ]:
from sagemaker.core.resources import Model
from sagemaker.core.shapes import (ContainerDefinition, ModelDataSource, S3ModelDataSource)

try:
    Model.get(model_name)
    print(f"model exists: {model_name}")
except Exception:
    Model.create(
        model_name=model_name,
        primary_container=ContainerDefinition(
            image=inference_image,
            model_data_source=ModelDataSource(
                s3_data_source=S3ModelDataSource(
                    s3_uri=merged_model_s3_uri, s3_data_type="S3Prefix",
                    compression_type="None")),
            environment=env),
        execution_role_arn=role)
    print(f"created model: {model_name}")

### Create the inference component

In [ ]:
from sagemaker.core.resources import InferenceComponent
from sagemaker.core.shapes import (InferenceComponentComputeResourceRequirements,
                                   InferenceComponentRuntimeConfig,
                                   InferenceComponentSpecification)

try:
    ic = InferenceComponent.get(ic_name)
    print(f"inference component exists: {ic_name}")
except Exception:
    ic = InferenceComponent.create(
        inference_component_name=ic_name,
        endpoint_name=endpoint_name,
        variant_name="AllTraffic",
        specification=InferenceComponentSpecification(
            model_name=model_name,
            compute_resource_requirements=InferenceComponentComputeResourceRequirements(
                # The component only sees the devices it reserves, so this has to
                # match SM_VLLM_TENSOR_PARALLEL_SIZE above: asking for fewer GPUs
                # than the tensor parallel size makes vLLM fail to initialise.
                min_memory_required_in_mb=1024 * 96,
                number_of_accelerator_devices_required=number_of_gpu)),
        runtime_config=InferenceComponentRuntimeConfig(copy_count=1),
        region=region)
    print(f"creating inference component: {ic_name}")

ic.wait_for_status("InService")

TUNED_MODEL_ID = f"sm:{endpoint_name}/{ic_name}@{region}"
print(f"\nready. serving model id:\n  {TUNED_MODEL_ID}")

### Smoke test on a real contract

The request is built by `C.build_messages(doc, labels)`, imported rather than rebuilt
here — the point of keeping the prompt in one module is that a served request cannot
drift from the module every other notebook reads.

One difference is worth naming rather than glossing over. Notebook 1 trains on
`C.build_prompt`, a **single string** with the contract before the checklist. This
endpoint speaks the OpenAI chat schema, so it needs **turns**, which is what
`build_messages` returns: the checklist in a system turn, the contract in the user turn.
Same instruction, same `/no_think`, different order — so this is not the byte-identical
prompt the model was trained on.

That is a deliberate trade, not an oversight. A chat endpoint wants a system turn, and
the instruction is explicit enough to survive the reordering. But it is exactly the kind
of difference that silently costs a fine-tune its gains, so the cell below checks that
the shared scorer can extract a JSON object and count expected keys, without validating item schemas. If you
would rather serve the trained string verbatim, send a single user turn containing
`C.build_prompt(doc, labels)` and nothing else.

In [ ]:
import json
import time

import boto3
from botocore.config import Config

import contractnli as C

C.ensure_dataset("./data")
test_docs, labels = C.load("test")
doc = test_docs[0]

# Raw SageMaker runtime call — this is the request an application would make.
# The vLLM container speaks the OpenAI chat schema, and the inference component is
# selected with its own header, not inside the body.
smr = boto3.client(
    "sagemaker-runtime",
    region_name=region,
    config=Config(read_timeout=300, retries={"total_max_attempts": 3}),
)


def ask_endpoint(doc, max_tokens=4000):
    """Invoke the deployed model and return (text, usage).

    The turns come from `C.build_messages`, so the prompt is imported rather than
    rebuilt: pasting a second copy into a notebook is how train/serve skew gets
    introduced. Note that this is the two-turn form (checklist in the system turn), while
    notebook 1 trains on the single-string `C.build_prompt` — see the markdown above for
    why that is safe here and how to serve the trained string verbatim instead.
    """
    body = {
        "model_name": ic_name,
        "messages": [{"role": m["role"],
                      "content": [{"type": "text", "text": m["content"]}]}
                     for m in C.build_messages(doc, labels)],
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "stop": ["<|im_end|>"],
        "stream": False,
    }
    response = smr.invoke_endpoint(
        EndpointName=endpoint_name,
        InferenceComponentName=ic_name,
        ContentType="application/json",
        Body=json.dumps(body),
    )
    payload = json.loads(response["Body"].read())
    text = payload["choices"][0]["message"]["content"]
    usage = payload.get("usage", {})
    return text, usage


from contractnli_scorer import parse_answer

t0 = time.time()
text, usage = ask_endpoint(doc)
elapsed = time.time() - t0

pred, ok = parse_answer(text)
covered = set(pred or {}) & set(labels)
print(f"first call {elapsed:.1f}s | parseable object: {ok} | usage: {usage}")
print(f"covered {len(covered)} of {len(labels)} checklist keys; schema not validated")

# Serving check only — no accuracy here. The base and fine-tuned model are scored
# exclusively by the managed evaluation pipeline in notebook 3, so the lab reports
# one set of numbers produced one way.


### Clean up

> **Important:** an endpoint bills per instance-hour for as long as it exists,
> whether or not you send traffic. On the reference run, an idle GPU box cost
> more than **twice** all the fine-tuning combined. Delete it when done.

Delete the inference component first, then wait before deleting the endpoint.

In [ ]:
import time

for label, fn in [
    ("inference component", lambda: InferenceComponent.get(ic_name).delete()),
    ("endpoint", lambda: Endpoint.get(endpoint_name).delete()),
    ("endpoint config", lambda: EndpointConfig.get(endpoint_config_name).delete()),
    ("model", lambda: Model.get(model_name).delete()),
]:
    try:
        fn()
        print(f"deleted {label}")
    except Exception as e:
        print(f"skip {label}: {type(e).__name__}")
    if label == "inference component":
        time.sleep(45)